In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


The API key found and looks good so far!


# LangChain

Chapter 11's listing is an autonomous coding agent: it writes Python for a use case, reviews its own output against a list of goals, and keeps refining until an LLM judge answers `True`. The book's framing (p. 165):

> It employs a "goal-setting and monitoring" pattern where it doesn't just generate code once, but enters into an iterative cycle of creation, self-evaluation, and improvement. The agent's success is measured by its own AI-driven judgment on whether the generated code successfully meets the initial objectives.

Three model calls per iteration, and each has a different job: **generate** the code, **critique** it against the goals, then **judge** whether the critique says the goals are met. Only the third one controls the loop.

**Ported from the book** (Mahtab Syed's MIT-licensed listing, kept otherwise intact):

| The book | Here |
| --- | --- |
| `if __name__ == "__main__":` | a direct call — the notebook is already `__main__` |
| `Path.cwd()` for the output file | `notebooks/generated/`, so a run does not drop a `.py` file into the repository root |
| `model="gpt-4o"` | `gpt-4o-mini`, which the listing's own comment invites; the loop can make up to 16 calls |
| emoji in every `print` | dropped, as everywhere else in these notebooks |
| — | one added line: the monitor's raw verdict is printed. The cell below is the reason |

In [3]:
# MIT License
# Copyright (c) 2025 Mahtab Syed
# https://www.linkedin.com/in/mahtabsyed/
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so.

"""
Hands-On Code Example - Iteration 2
- To illustrate the Goal Setting and Monitoring pattern, we have an example using LangChain and OpenAI APIs:

Objective: Build an AI Agent which can write code for a specified use case based on specified goals:
- Accepts a coding problem (use case) in code or can be as input.
- Accepts a list of goals (e.g., "simple", "tested", "handles edge cases")  in code or can be input.
- Uses an LLM (like GPT-4o) to generate and refine Python code until the goals are met. (I am using max 5 iterations, this could be based on a set goal as well)
- To check if we have met our goals I am asking the LLM to judge this and answer just True or False which makes it easier to stop the iterations.
- Saves the final code in a .py file with a clean filename and a header comment.
"""

# Ported for this notebook: the book's __main__ guard is replaced by a direct call, the
# generated file is written to notebooks/generated/ instead of the working directory, and
# the emoji in the print statements are dropped. gpt-4o-mini stands in for gpt-4o, which
# the listing's own comment invites; the loop below can make up to 16 model calls.

import os
import random
import re
from pathlib import Path
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv, find_dotenv

# Load environment variables
_ = load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise EnvironmentError("Please set the OPENAI_API_KEY environment variable.")

# Where the finished program is written. The book used the working directory, which for
# a notebook is notebooks/, so every run would drop a file into the repository.
OUTPUT_DIR = Path.cwd() / "generated"
OUTPUT_DIR.mkdir(exist_ok=True)

# Initialize OpenAI model
print("Initializing OpenAI LLM (gpt-4o-mini)...")
llm = ChatOpenAI(
    model="gpt-4o-mini",  # the book uses gpt-4o; any OpenAI chat model works
    temperature=0.3,
    openai_api_key=OPENAI_API_KEY,
)

# --- Utility Functions ---

def generate_prompt(
    use_case: str, goals: list[str], previous_code: str = "", feedback: str = ""
) -> str:
    print("Constructing prompt for code generation...")
    base_prompt = f"""
You are an AI coding agent. Your job is to write Python code based on the following use case:

Use Case: {use_case}

Your goals are:
{chr(10).join(f"- {g.strip()}" for g in goals)}
"""
    if previous_code:
        print("Adding previous code to the prompt for refinement.")
        base_prompt += f"\nPreviously generated code:\n{previous_code}"
    if feedback:
        print("Including feedback for revision.")
        base_prompt += f"\nFeedback on previous version:\n{feedback}\n"

    base_prompt += "\nPlease return only the revised Python code. Do not include comments or explanations outside the code."
    return base_prompt

def get_code_feedback(code: str, goals: list[str]) -> str:
    print("Evaluating code against the goals...")
    feedback_prompt = f"""
You are a Python code reviewer. A code snippet is shown below. Based on the following goals:

{chr(10).join(f"- {g.strip()}" for g in goals)}

Please critique this code and identify if the goals are met. Mention if improvements are needed for clarity, simplicity, correctness, edge case handling, or test coverage.

Code:
{code}
"""
    return llm.invoke(feedback_prompt)

def goals_met(feedback_text: str, goals: list[str]) -> bool:
    """
    Uses the LLM to evaluate whether the goals have been met based on the feedback text.
    Returns True or False (parsed from LLM output).
    """
    review_prompt = f"""
You are an AI reviewer.

Here are the goals:
{chr(10).join(f"- {g.strip()}" for g in goals)}

Here is the feedback on the code:
\"\"\"
{feedback_text}
\"\"\"

Based on the feedback above, have the goals been met?

Respond with only one word: True or False.
"""
    response = llm.invoke(review_prompt).content.strip().lower()
    # The book compares to "true" exactly. A model that answers "True." or explains
    # itself never stops the loop, so the monitor is only as good as this one line.
    print(f"  monitor verdict: {response!r}")
    return response == "true"

def clean_code_block(code: str) -> str:
    lines = code.strip().splitlines()
    if lines and lines[0].strip().startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    return "\n".join(lines).strip()

def add_comment_header(code: str, use_case: str) -> str:
    comment = f"# This Python program implements the following use case:\n# {use_case.strip()}\n"
    return comment + "\n" + code

def to_snake_case(text: str) -> str:
    text = re.sub(r"[^a-zA-Z0-9 ]", "", text)
    return re.sub(r"\s+", "_", text.strip().lower())

def save_code_to_file(code: str, use_case: str) -> str:
    print("Saving final code to file...")

    summary_prompt = (
        f"Summarize the following use case into a single lowercase word or phrase, "
        f"no more than 10 characters, suitable for a Python filename:\n\n{use_case}"
    )
    raw_summary = llm.invoke(summary_prompt).content.strip()
    short_name = re.sub(r"[^a-zA-Z0-9_]", "", raw_summary.replace(" ", "_").lower())[:10]

    random_suffix = str(random.randint(1000, 9999))
    filename = f"{short_name}_{random_suffix}.py"
    filepath = OUTPUT_DIR / filename

    with open(filepath, "w") as f:
        f.write(code)

    print(f"Code saved to: {filepath}")
    return str(filepath)

# --- Main Agent Function ---

def run_code_agent(use_case: str, goals_input: str, max_iterations: int = 5) -> str:
    goals = [g.strip() for g in goals_input.split(",")]

    print(f"\nUse Case: {use_case}")
    print("Goals:")
    for g in goals:
        print(f"  - {g}")

    previous_code = ""
    feedback = ""

    for i in range(max_iterations):
        print(f"\n=== Iteration {i + 1} of {max_iterations} ===")
        prompt = generate_prompt(use_case, goals, previous_code, feedback if isinstance(feedback, str) else feedback.content)

        print("Generating code...")
        code_response = llm.invoke(prompt)
        raw_code = code_response.content.strip()
        code = clean_code_block(raw_code)
        print("\nGenerated Code:\n" + "-" * 50 + f"\n{code}\n" + "-" * 50)

        print("\nSubmitting code for feedback review...")
        feedback = get_code_feedback(code, goals)
        feedback_text = feedback.content.strip()
        print("\nFeedback Received:\n" + "-" * 50 + f"\n{feedback_text}\n" + "-" * 50)

        if goals_met(feedback_text, goals):
            print("LLM confirms goals are met. Stopping iteration.")
            break

        print("Goals not fully met. Preparing for next iteration...")
        previous_code = code

    final_code = add_comment_header(code, use_case)
    return save_code_to_file(final_code, use_case)

# --- Test Run ---
# The book wrapped this in `if __name__ == "__main__":`, with two more examples
# commented out below it.

print("\nWelcome to the AI Code Generation Agent")

use_case_input = "Write code to find BinaryGap of a given positive integer"
goals_input = "Code simple to understand, Functionally correct, Handles comprehensive edge cases, Takes positive integer input only, prints the results with few examples"
run_code_agent(use_case_input, goals_input)

Initializing OpenAI LLM (gpt-4o-mini)...

Welcome to the AI Code Generation Agent

Use Case: Write code to find BinaryGap of a given positive integer
Goals:
  - Code simple to understand
  - Functionally correct
  - Handles comprehensive edge cases
  - Takes positive integer input only
  - prints the results with few examples

=== Iteration 1 of 5 ===
Constructing prompt for code generation...
Generating code...



Generated Code:
--------------------------------------------------
def binary_gap(n):
    if n <= 0:
        raise ValueError("Input must be a positive integer.")
    
    binary = bin(n)[2:]
    max_gap = 0
    current_gap = 0
    found_one = False

    for bit in binary:
        if bit == '1':
            if found_one:
                max_gap = max(max_gap, current_gap)
            found_one = True
            current_gap = 0
        elif found_one:
            current_gap += 1

    return max_gap

# Example usage
print(binary_gap(9))    # Output: 2 (binary representation: 1001)
print(binary_gap(20))   # Output: 1 (binary representation: 10100)
print(binary_gap(15))   # Output: 0 (binary representation: 1111)
print(binary_gap(32))   # Output: 0 (binary representation: 100000)
print(binary_gap(1041)) # Output: 5 (binary representation: 10000010001)
--------------------------------------------------

Submitting code for feedback review...
Evaluating code against the goals...



Feedback Received:
--------------------------------------------------
The provided code snippet implements a function to calculate the maximum binary gap of a positive integer. Let's evaluate the code against the specified goals:

### Goals Evaluation

1. **Code Simple to Understand**:
   - The code is relatively straightforward. It converts the integer to its binary representation and iterates through the bits to calculate the maximum gap. However, adding comments or docstrings would enhance clarity for readers unfamiliar with the logic.

2. **Functionally Correct**:
   - The function correctly computes the maximum binary gap, as demonstrated by the provided examples. It handles the logic for counting gaps between '1's effectively.

3. **Handles Comprehensive Edge Cases**:
   - The code raises a `ValueError` for non-positive integers, which is good. However, it does not handle the case where the input is `1`, which should return `0` since there are no gaps in the binary representatio

  monitor verdict: 'true'
LLM confirms goals are met. Stopping iteration.
Saving final code to file...


Code saved to: /Users/hodung/Workspace/ta/agentic-design-patterns/notebooks/generated/binarygap_6556.py


'/Users/hodung/Workspace/ta/agentic-design-patterns/notebooks/generated/binarygap_6556.py'

## LangChain + Ollama

In [4]:
# Same goal-driven code agent, running on a local model served by Ollama.
# Prerequisites:
#   ollama pull llama3.1:8b
# Reuses every function from the cell above: they all read the module-level `llm`, so
# rebinding it is the whole port. max_iterations is lowered because each iteration costs
# three local calls, two of them over a whole program.
#
# Watch the monitor verdict, not the code. This local model answers "true." with a
# trailing period, and the book's goals_met does `response == "true"`, so every verdict
# is read as a rejection: the agent is told its goals are met three times and refines
# anyway until it hits the cap. The goal setting works; the monitoring is one brittle
# string comparison. `response.startswith("true")` would honour it - left unfixed here
# because the failure is the most instructive thing in the chapter.
from langchain_ollama import ChatOllama

OLLAMA_MODEL = "llama3.1:8b"  # any model from `ollama list`, e.g. qwen2.5:7b

print(f"Initializing Ollama LLM ({OLLAMA_MODEL})...")
llm = ChatOllama(model=OLLAMA_MODEL, temperature=0.3)

run_code_agent(use_case_input, goals_input, max_iterations=3)

Initializing Ollama LLM (llama3.1:8b)...

Use Case: Write code to find BinaryGap of a given positive integer
Goals:
  - Code simple to understand
  - Functionally correct
  - Handles comprehensive edge cases
  - Takes positive integer input only
  - prints the results with few examples

=== Iteration 1 of 3 ===
Constructing prompt for code generation...
Generating code...



Generated Code:
--------------------------------------------------
def binary_gap(N):
    if N <= 0:
        return 0
    max_gap = 0
    prev_1 = None
    while N > 0:
        if N % 2 == 1:
            if prev_1 is not None:
                max_gap = max(max_gap, N - prev_1)
            prev_1 = N
        N //= 2
    return max_gap

print(binary_gap(5))  # Output: 2
print(binary_gap(6))  # Output: 1
print(binary_gap(8))  # Output: 0
print(binary_gap(12))  # Output: 2
print(binary_gap(20))  # Output: 1
--------------------------------------------------

Submitting code for feedback review...
Evaluating code against the goals...



Feedback Received:
--------------------------------------------------
**Code Review**

The provided code snippet calculates the maximum gap between consecutive 1's in the binary representation of a given positive integer. Here's a critique of the code:

**Goals Met:**

1. **Code simplicity**: The code is straightforward and easy to follow.
2. **Functionally correct**: The code correctly calculates the maximum gap between consecutive 1's in the binary representation of the input number.
3. **Handles comprehensive edge cases**: The code checks for invalid inputs (N <= 0) and handles them correctly.
4. **Takes positive integer input only**: The code assumes that the input will be a positive integer, but it does not explicitly check for this. We'll discuss this further below.
5. **Prints the results with few examples**: The code includes example usage and prints the results, which is helpful for understanding the function's behavior.

**Improvement Suggestions:**

1. **Input validation**:

  monitor verdict: 'true.'
Goals not fully met. Preparing for next iteration...

=== Iteration 2 of 3 ===
Constructing prompt for code generation...
Adding previous code to the prompt for refinement.
Including feedback for revision.
Generating code...



Generated Code:
--------------------------------------------------
def binary_gap(N):
    """
    Calculate the maximum gap between consecutive 1's in the binary representation of N.

    Args:
        N (int): A positive integer.

    Returns:
        int: The maximum gap between consecutive 1's.

    Raises:
        ValueError: If N is not a positive integer.
    """
    if not isinstance(N, int) or N <= 0:
        raise ValueError("Input must be a positive integer")

    max_gap = 0
    prev_1 = None
    while N > 0:
        if N % 2 == 1:
            if prev_1 is not None:
                max_gap = max(max_gap, N - prev_1)
            prev_1 = N
        N //= 2
    return max_gap

# Example usage:
print(binary_gap(5))  # Output: 2
print(binary_gap(6))  # Output: 1
print(binary_gap(8))  # Output: 0
print(binary_gap(12))  # Output: 2
print(binary_gap(20))  # Output: 1
--------------------------------------------------

Submitting code for feedback review...
Evaluating code against t

...

  monitor verdict: 'true.'
Goals not fully met. Preparing for next iteration...

=== Iteration 3 of 3 ===
Constructing prompt for code generation...
Adding previous code to the prompt for refinement.
Including feedback for revision.
Generating code...



Generated Code:
--------------------------------------------------
def max_binary_gap(N):
    """
    Calculate the maximum gap between consecutive 1's in the binary representation of N.

    Args:
        N (int): A positive integer.

    Returns:
        int: The maximum gap between consecutive 1's.

    Raises:
        ValueError: If N is not a positive integer.
    """
    if not isinstance(N, int) or N <= 0:
        raise ValueError("Input must be a positive integer")

    max_gap = 0
    prev_1 = None
    binary_str = bin(N)[2:]  # Get the binary representation as a string
    for i, bit in enumerate(binary_str):
        if bit == '1':
            if prev_1 is not None:
                max_gap = max(max_gap, i - prev_1)
            prev_1 = i
    return max_gap

# Example usage:
print(max_binary_gap(5))  # Output: 2
print(max_binary_gap(6))  # Output: 1
print(max_binary_gap(8))  # Output: 0
print(max_binary_gap(12))  # Output: 2
print(max_binary_gap(20))  # Output: 1
print(max_b


Feedback Received:
--------------------------------------------------
**Code Review**

The code is generally well-structured and easy to understand. However, there are some improvements that can be made to enhance its simplicity, correctness, and edge case handling.

**Goals Met:**

1. **Code is simple to understand**: The code is well-documented, and the variable names are descriptive.
2. **Functionally correct**: The code correctly calculates the maximum gap between consecutive 1's in the binary representation of N.
3. **Handles comprehensive edge cases**: The code raises a `ValueError` if N is not a positive integer, and it correctly handles the case where there are no consecutive 1's in the binary representation.

**Improvements Needed:**

1. **Edge case handling**: While the code raises a `ValueError` if N is not a positive integer, it does not handle the case where N is 0. In this case, the function should return 0, as there are no gaps between consecutive 1's.
2. **Input valida

  monitor verdict: 'true.'
Goals not fully met. Preparing for next iteration...
Saving final code to file...
Code saved to: /Users/hodung/Workspace/ta/agentic-design-patterns/notebooks/generated/binarygap_5057.py


'/Users/hodung/Workspace/ta/agentic-design-patterns/notebooks/generated/binarygap_5057.py'